# NinaPro EMG Explorer

Load a specific subject from a NinaPro dataset and inspect the data.

In [ ]:
## Configuration — edit these two values
DB = 2        # dataset number (1–5)
SUBJECT = 1   # subject number

In [ ]:
import glob
from pathlib import Path

import numpy as np
import scipy.io as sio

NINAPRO_DIR = Path("ninapro")

def load_subject(db: int, subject: int, ninapro_dir: Path = NINAPRO_DIR) -> dict[str, np.ndarray]:
    """Load all exercise .mat files for a subject and concatenate them.

    Returns a dict with keys: emg, stimulus, restimulus, repetition.
    All arrays are (n_samples, n_channels) or (n_samples,) for labels.
    """
    subject_dir = ninapro_dir / f"DB{db}" / f"DB{db}_s{subject}"
    # mat files are named e.g. S1_E1_A1.mat (DB1) or DB2_s1_E1_A1.mat
    patterns = [
        str(subject_dir / f"S{subject}_E*_A*.mat"),
        str(subject_dir / f"s{subject}_E*_A*.mat"),
        str(subject_dir / f"DB{db}_s{subject}_E*_A*.mat"),
        str(subject_dir / f"DB{db}_S{subject}_E*_A*.mat"),
    ]
    mat_files = []
    for p in patterns:
        mat_files.extend(sorted(glob.glob(p)))

    if not mat_files:
        raise FileNotFoundError(
            f"No .mat files found for DB{db} subject {subject} in {subject_dir}.\n"
            f"Run: python get_ninapro.py --db {db} --subjects {subject}\n"
            f"Current working directory: {Path.cwd()}"
        )

    arrays: dict[str, list] = {"emg": [], "stimulus": [], "restimulus": [], "repetition": []}
    for path in mat_files:
        mat = sio.loadmat(path, squeeze_me=True)
        for key in arrays:
            if key in mat:
                arrays[key].append(np.atleast_1d(mat[key]))

    return {k: np.concatenate(v) if v else np.array([]) for k, v in arrays.items()}


data = load_subject(DB, SUBJECT)
print(f"Loaded DB{DB} subject {SUBJECT}")
for key, arr in data.items():
    print(f"  {key:12s}: shape={arr.shape}, dtype={arr.dtype}")

In [ ]:
import matplotlib.pyplot as plt

emg = data["emg"]
stimulus = data["stimulus"]
n_channels = emg.shape[1] if emg.ndim == 2 else 1
t = np.arange(len(emg))

fig, axes = plt.subplots(n_channels + 1, 1, figsize=(14, 2 * (n_channels + 1)), sharex=True)

for ch in range(n_channels):
    axes[ch].plot(t, emg[:, ch] if emg.ndim == 2 else emg, lw=0.4)
    axes[ch].set_ylabel(f"Ch {ch+1}")

axes[-1].plot(t, stimulus, color="tab:red", lw=0.8)
axes[-1].set_ylabel("Stimulus")
axes[-1].set_xlabel("Sample")

fig.suptitle(f"DB{DB} — Subject {SUBJECT}", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Unique classes (excluding rest=0)
classes = np.unique(stimulus[stimulus > 0])
print("Movement classes:", classes)

## Labeled window extraction

Slide a fixed-length window over the EMG signal and assign the majority label within each window.
`restimulus` (re-labeled after the experiment) is used instead of `stimulus` for cleaner boundaries.

In [ ]:
from collections import Counter

WINDOW_SIZE = 200   # samples  (e.g. 200 @ 2 kHz = 100 ms)
WINDOW_STEP = 100   # overlap stride

def extract_windows(
    emg: np.ndarray,
    labels: np.ndarray,
    window_size: int = WINDOW_SIZE,
    step: int = WINDOW_STEP,
    drop_rest: bool = True,
) -> tuple[np.ndarray, np.ndarray]:
    """Return (windows, labels) arrays.

    windows : (N, window_size, n_channels)
    labels  : (N,)  majority label within each window; windows where the
               majority label is 0 (rest) are dropped when drop_rest=True.
    """
    n_samples, n_channels = emg.shape
    starts = range(0, n_samples - window_size + 1, step)

    X, y = [], []
    for s in starts:
        window_labels = labels[s : s + window_size]
        majority = Counter(window_labels.tolist()).most_common(1)[0][0]
        if drop_rest and majority == 0:
            continue
        X.append(emg[s : s + window_size])
        y.append(majority)

    return np.stack(X).astype(np.float32), np.array(y, dtype=np.int64)


labels = data["restimulus"] if data["restimulus"].size else data["stimulus"]
X, y = extract_windows(data["emg"], labels)

print(f"Windows : {X.shape}   (N, window_size, channels)")
print(f"Labels  : {y.shape}")
print(f"Classes : {np.unique(y)}")
print(f"Distribution: { {int(k): int(v) for k, v in Counter(y.tolist()).most_common()} }")

## PyTorch Dataset & DataLoader

`NinaProDataset` loads one subject lazily (the `.mat` files are read once on first use) and serves windows on-the-fly via `__getitem__` — no giant pre-allocated tensor needed.

Set `subjects` to a list to combine multiple subjects into one dataset.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, ConcatDataset


class NinaProDataset(Dataset):
    """Lazy-loading windowed EMG dataset for a single NinaPro subject.

    The subject's .mat files are loaded once on first __getitem__ call.
    Windows are sliced on-the-fly, so only one subject's raw signal
    sits in memory at a time.

    Args:
        db:          Dataset number (1–5).
        subject:     Subject number.
        window_size: Window length in samples.
        step:        Stride between consecutive windows.
        drop_rest:   If True, windows whose majority label is 0 are excluded.
        ninapro_dir: Root directory containing DB{N}/ subdirectories.
        label_map:   Optional dict mapping original label ints to new ints.
                     Useful for remapping class indices to 0-based contiguous.
    """

    def __init__(
        self,
        db: int,
        subject: int,
        window_size: int = 200,
        step: int = 100,
        drop_rest: bool = True,
        ninapro_dir: Path = NINAPRO_DIR,
        label_map: dict[int, int] | None = None,
    ):
        self.db = db
        self.subject = subject
        self.window_size = window_size
        self.step = step
        self.drop_rest = drop_rest
        self.ninapro_dir = ninapro_dir
        self.label_map = label_map

        self._emg: np.ndarray | None = None
        self._labels: np.ndarray | None = None
        self._indices: list[int] | None = None  # valid window start positions

    def _load(self) -> None:
        data = load_subject(self.db, self.subject, self.ninapro_dir)
        self._emg = data["emg"].astype(np.float32)
        raw_labels = data["restimulus"] if data["restimulus"].size else data["stimulus"]

        valid_starts = []
        for s in range(0, len(self._emg) - self.window_size + 1, self.step):
            window_labels = raw_labels[s : s + self.window_size]
            majority = Counter(window_labels.tolist()).most_common(1)[0][0]
            if self.drop_rest and majority == 0:
                continue
            valid_starts.append(s)

        self._labels = raw_labels
        self._indices = valid_starts

    @property
    def emg(self) -> np.ndarray:
        if self._emg is None:
            self._load()
        return self._emg

    def __len__(self) -> int:
        if self._indices is None:
            self._load()
        return len(self._indices)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        if self._indices is None:
            self._load()
        start = self._indices[idx]
        window = self._emg[start : start + self.window_size]          # (W, C)
        majority = Counter(
            self._labels[start : start + self.window_size].tolist()
        ).most_common(1)[0][0]
        label = self.label_map[majority] if self.label_map else majority
        return torch.from_numpy(window), torch.tensor(label, dtype=torch.long)


# --- build label map so class indices are 0-based contiguous ---
# (NinaPro labels start at 1 for movements; 0 = rest, which is dropped)
all_labels = sorted(int(c) for c in np.unique(y))   # from the extracted windows above
label_map = {orig: new for new, orig in enumerate(all_labels)}
print("Label map (original → 0-based):", label_map)

dataset = NinaProDataset(
    db=DB,
    subject=SUBJECT,
    window_size=WINDOW_SIZE,
    step=WINDOW_STEP,
    label_map=label_map,
)
print(f"Dataset size: {len(dataset)} windows")

# Example: combine multiple subjects
# dataset = ConcatDataset([
#     NinaProDataset(DB, s, label_map=label_map) for s in [1, 2, 3]
# ])

loader = DataLoader(dataset, batch_size=64, shuffle=True, num_workers=0)

batch_x, batch_y = next(iter(loader))
print(f"Batch EMG  : {batch_x.shape}   (batch, window_size, channels)")
print(f"Batch labels: {batch_y.shape},  unique={batch_y.unique().tolist()}")

## Preprocessing & Classification Pipeline

Literature-grounded choices:
- **Bandpass 10–500 Hz** (4th-order Butterworth): removes DC drift and high-freq noise [Atzori et al. 2016, Côté-Allard et al. 2019]
- **Per-channel z-score normalisation** computed on the training set only, applied to both splits
- **Train/test split by repetition**: reps 1–4 → train, reps 5–6 → test. This is the standard protocol to avoid leakage from overlapping windows [Atzori et al. 2016]
- **1D multi-scale CNN**: parallel branches with kernel sizes 3 / 7 / 15 capture short and long temporal patterns; features are concatenated then refined — similar structure to Hu et al. (2018) and the multi-scale attention networks (2023)
- **BatchNorm + Dropout 0.5** after each block [standard across literature]
- **Adam + cosine annealing LR** [Côté-Allard et al. 2019, EMGBench 2024]

In [ ]:
from scipy.signal import butter, sosfiltfilt

FS = 2000  # Hz — NinaPro DB2–DB5 sampling rate (DB1: 100 Hz, adjust if needed)

def bandpass_filter(emg: np.ndarray, fs: int = FS, low: float = 10.0, high: float = 500.0) -> np.ndarray:
    """4th-order Butterworth bandpass, applied zero-phase (forward + backward)."""
    sos = butter(4, [low, high], btype="bandpass", fs=fs, output="sos")
    return sosfiltfilt(sos, emg, axis=0).astype(np.float32)


class NinaProDatasetByRepetition(Dataset):
    """Dataset that splits by repetition index to prevent window-overlap leakage.

    train_reps / test_reps are lists of repetition numbers as stored in the
    'repetition' array of the .mat files (typically 1–6 for DB2).

    Preprocessing applied:
      1. Bandpass filter on the raw signal (fit-once, applied per-channel)
      2. Per-channel z-score normalisation — stats computed on train reps only,
         then passed in via `channel_stats` for the test set.
    """

    def __init__(
        self,
        db: int,
        subject: int,
        reps: list[int],
        window_size: int = 400,      # 200 ms @ 2 kHz
        step: int = 40,              # 20 ms stride — tight overlap for more samples
        drop_rest: bool = True,
        ninapro_dir: Path = NINAPRO_DIR,
        label_map: dict[int, int] | None = None,
        channel_stats: tuple[np.ndarray, np.ndarray] | None = None,
        fs: int = FS,
    ):
        self.window_size = window_size
        self.step = step
        self.label_map = label_map

        raw = load_subject(db, subject, ninapro_dir)
        emg = raw["emg"].astype(np.float32)
        labels = (raw["restimulus"] if raw["restimulus"].size else raw["stimulus"])
        repetition = raw["repetition"].astype(int)

        # --- bandpass filter ---
        emg = bandpass_filter(emg, fs=fs)

        # --- keep only requested repetitions ---
        mask = np.isin(repetition, reps)
        emg, labels = emg[mask], labels[mask]

        # --- per-channel z-score ---
        if channel_stats is None:
            self.mean = emg.mean(axis=0, keepdims=True)
            self.std  = emg.std(axis=0, keepdims=True).clip(min=1e-8)
        else:
            self.mean, self.std = channel_stats
        emg = (emg - self.mean) / self.std

        # --- build valid window index ---
        valid_starts = []
        for s in range(0, len(emg) - window_size + 1, step):
            win_labels = labels[s : s + window_size]
            majority = Counter(win_labels.tolist()).most_common(1)[0][0]
            if drop_rest and majority == 0:
                continue
            valid_starts.append((s, majority))

        self._emg = emg
        self._windows = valid_starts  # list of (start, label)

    @property
    def channel_stats(self) -> tuple[np.ndarray, np.ndarray]:
        return self.mean, self.std

    def __len__(self) -> int:
        return len(self._windows)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        start, majority = self._windows[idx]
        window = self._emg[start : start + self.window_size]  # (W, C)
        label = self.label_map[majority] if self.label_map else majority
        # CNN expects (C, W) — channels-first
        return torch.from_numpy(window.T), torch.tensor(label, dtype=torch.long)


# Build train/test split by repetition
TRAIN_REPS = [1, 2, 3, 4]
TEST_REPS  = [5, 6]
WINDOW_SIZE_CLS = 400   # 200 ms @ 2 kHz
WINDOW_STEP_CLS = 40    # 20 ms stride

train_ds = NinaProDatasetByRepetition(
    DB, SUBJECT, TRAIN_REPS,
    window_size=WINDOW_SIZE_CLS, step=WINDOW_STEP_CLS,
    label_map=label_map,
)
test_ds = NinaProDatasetByRepetition(
    DB, SUBJECT, TEST_REPS,
    window_size=WINDOW_SIZE_CLS, step=WINDOW_STEP_CLS,
    label_map=label_map,
    channel_stats=train_ds.channel_stats,  # use train stats — no leakage
)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False, num_workers=0, pin_memory=True)

print(f"Train windows : {len(train_ds)}")
print(f"Test  windows : {len(test_ds)}")
x0, y0 = train_ds[0]
print(f"Window shape  : {x0.shape}  (channels, time)")
print(f"Num classes   : {len(label_map)}")

### Model: Multi-Scale 1D CNN

Three parallel branches with kernel sizes **3 / 7 / 15** (≈1.5 / 3.5 / 7.5 ms at 2 kHz) capture local, mid-range, and coarser temporal patterns independently. Their feature maps are concatenated and refined by a shared conv block before global average pooling → classifier head.

Global average pooling instead of a large FC layer dramatically reduces parameters and acts as an implicit regulariser [Lin et al., NiN 2014; standard in EMG literature].

```
Input (C, W)
  ├─ Conv1d k=3  → BN → ReLU ─┐
  ├─ Conv1d k=7  → BN → ReLU ─┤ concat → Conv1d → BN → ReLU → Dropout
  └─ Conv1d k=15 → BN → ReLU ─┘
                                → GlobalAvgPool → Linear → num_classes
```

In [ ]:
import torch.nn as nn


class ConvBnRelu(nn.Sequential):
    def __init__(self, in_ch: int, out_ch: int, kernel: int, **kwargs):
        super().__init__(
            nn.Conv1d(in_ch, out_ch, kernel, padding=kernel // 2, bias=False, **kwargs),
            nn.BatchNorm1d(out_ch),
            nn.ReLU(inplace=True),
        )


class MultiScaleEMGNet(nn.Module):
    """Multi-scale 1D CNN for EMG gesture classification.

    Input : (batch, n_channels, window_size)
    Output: (batch, num_classes)  — raw logits

    Architecture inspired by:
      - Hu et al. 2018 (multi-scale temporal features)
      - Atzori et al. 2016 baseline (conv + pool + FC)
      - NiN / GAP pattern (no large FC, fewer params, less overfitting)
    """

    def __init__(self, n_channels: int, num_classes: int, dropout: float = 0.5):
        super().__init__()
        base = 64

        # Parallel branches — same number of filters, different receptive fields
        self.branch3  = ConvBnRelu(n_channels, base, kernel=3)
        self.branch7  = ConvBnRelu(n_channels, base, kernel=7)
        self.branch15 = ConvBnRelu(n_channels, base, kernel=15)

        # Fusion + deeper refinement
        self.fuse = nn.Sequential(
            ConvBnRelu(base * 3, 128, kernel=3),
            nn.MaxPool1d(2),
            ConvBnRelu(128, 128, kernel=3),
            nn.MaxPool1d(2),
            ConvBnRelu(128, 256, kernel=3),
            nn.Dropout(dropout),
        )

        # Global average pool collapses time dimension → no FC size dependency
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Linear(256, num_classes)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, C, W)
        b3  = self.branch3(x)
        b7  = self.branch7(x)
        b15 = self.branch15(x)
        x = torch.cat([b3, b7, b15], dim=1)  # (B, 3*base, W)
        x = self.fuse(x)                      # (B, 256, W')
        x = self.gap(x).squeeze(-1)           # (B, 256)
        return self.classifier(x)             # (B, num_classes)


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
n_channels = x0.shape[0]
num_classes = len(label_map)

model = MultiScaleEMGNet(n_channels=n_channels, num_classes=num_classes).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Device     : {DEVICE}")
print(f"Channels   : {n_channels}   Classes: {num_classes}")
print(f"Parameters : {n_params:,}")

# Quick shape check
with torch.no_grad():
    dummy = torch.zeros(2, n_channels, WINDOW_SIZE_CLS).to(DEVICE)
    out = model(dummy)
    print(f"Output shape: {out.shape}  (batch, num_classes)")

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR

EPOCHS     = 50
LR         = 1e-3
WEIGHT_DECAY = 1e-4   # L2 regularisation

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)


def run_epoch(loader, train: bool) -> tuple[float, float]:
    model.train(train)
    total_loss = correct = n = 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            loss = criterion(logits, y)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(y)
            correct += (logits.argmax(1) == y).sum().item()
            n += len(y)
    return total_loss / n, correct / n


history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    te_loss, te_acc = run_epoch(test_loader,  train=False)
    scheduler.step()

    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["test_loss"].append(te_loss)
    history["test_acc"].append(te_acc)

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{EPOCHS}  "
              f"train loss={tr_loss:.4f} acc={tr_acc:.3f}  |  "
              f"test  loss={te_loss:.4f} acc={te_acc:.3f}")

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

# --- learning curves ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
epochs = range(1, EPOCHS + 1)
ax1.plot(epochs, history["train_loss"], label="train")
ax1.plot(epochs, history["test_loss"],  label="test")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.legend(); ax1.set_title("Loss")

ax2.plot(epochs, history["train_acc"], label="train")
ax2.plot(epochs, history["test_acc"],  label="test")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy"); ax2.legend(); ax2.set_title("Accuracy")
plt.tight_layout(); plt.show()

# --- confusion matrix + per-class report ---
model.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for x, y in test_loader:
        preds = model(x.to(DEVICE)).argmax(1).cpu()
        all_preds.append(preds)
        all_true.append(y)

all_preds = torch.cat(all_preds).numpy()
all_true  = torch.cat(all_true).numpy()

inv_label_map = {v: k for k, v in label_map.items()}
class_names = [str(inv_label_map[i]) for i in range(num_classes)]

print(classification_report(all_true, all_preds, target_names=class_names, zero_division=0))

fig, ax = plt.subplots(figsize=(min(20, num_classes), min(18, num_classes)))
ConfusionMatrixDisplay.from_predictions(all_true, all_preds, display_labels=class_names, ax=ax, colorbar=False)
ax.set_title(f"Confusion matrix — DB{DB} Subject {SUBJECT}")
plt.tight_layout(); plt.show()